# Long-Range Sentence-Shuffle Dissociation on Llama-3.1-8B

**Purpose:** decompose the long-range order-specific marginal-context-benefit signal into a *discourse-chaining* component and a *within-sentence syntax* component.

**Method:** at each log-spaced context length, measure target perplexity under three prior-context conditions:
- `ordered`            — the actual prior token sequence (intact).
- `sentence_shuffled`  — within-sentence order preserved, sentences permuted.
- `token_shuffled`     — all prior tokens permuted (matches existing long-range protocol).

**Decomposition (per interval):**
- Total order-specific gap rate (existing paper):  `ord_marg − tok_shuf_marg`
- Within-sentence syntax component:                `sent_shuf_marg − tok_shuf_marg`
- Discourse-chaining component:                    `ord_marg − sent_shuf_marg`

**Predictions:**
- If long-range structure is purely local syntax: `ord ≈ sent_shuf > tok_shuf`. Discourse-chaining component vanishes at long range.
- If long-range structure is purely discourse chaining: `ord > sent_shuf ≈ tok_shuf`. Within-sentence component vanishes at long range.
- Most likely: partial dissociation — both components contribute, with discourse chaining dominating at long distances.

**Output:** `My Drive/LRTIA/Results/corpus_expansion_longrange_sentshuf/llama/<corpus_id>.json`. Each record has `context_lengths`, `ordered_ppl`, `sentence_shuffled_ppl`, `token_shuffled_ppl` arrays of equal length.

Uses the same cells, log-spaced ctx grid [0..1024], and target-fraction policy as `Corpus_Expansion_LongRange_Llama`, but selects target_start at the next sentence boundary at or after the fractional position so that all prior tokens belong to complete sentences.

In [ ]:
!pip install -q -U accelerate

import numpy as np
import json, math, os, gc, random, re, time
from pathlib import Path
from tqdm.auto import tqdm
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive/LRTIA')
BASE = DRIVE / 'Results/corpus_expansion_longrange_sentshuf/llama'
BASE.mkdir(parents=True, exist_ok=True)
TARGETS_PATH = DRIVE / 'Results/corpus_expansion/targets_llama.jsonl'
TOK_MANIFEST_PATH = DRIVE / 'Results/corpus_expansion/tokenized_manifest_llama.jsonl'

MODEL_NAME = 'unsloth/Meta-Llama-3.1-8B'

# Same protocol as Corpus_Expansion_LongRange_Llama.
CTX_LENGTHS = [0, 1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]
MAX_CTX = max(CTX_LENGTHS)
TARGET_LEN = 30
TARGET_FRACS = [0.5]
MIN_DOC_TOK = MAX_CTX + TARGET_LEN + 50
N_SHUFFLES = 1
SEED = 20260503

# Same cells as the existing long-range run for direct comparison.
RUN_CORPORA = [
    'gutenberg_fiction_en',
    'ted_transcripts_en',
    'ted_transcripts_de',
    'literary_ja',
    'literary_fi',
    'news_en',
    'ted_transcripts_fr',
    'ted_transcripts_tr',
    'buckeye',
]

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Probe: {MODEL_NAME}')
print(f'Context lengths: {CTX_LENGTHS}')
print(f'Min doc length: {MIN_DOC_TOK} tokens')
print(f'Cells: {RUN_CORPORA}')

In [ ]:
# Load model — fp16 for H100 (skip 4-bit quant; dequant overhead hurts on H100).
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
)
model.eval()
print(f'{MODEL_NAME} loaded (fp16)')

In [ ]:
# ---- ppl pipeline (identical to Corpus_Expansion_LongRange_Llama) ----
@torch.no_grad()
def ppl_nll(ctx_toks, tgt_toks):
    if len(tgt_toks) < 2:
        return float('inf'), float('inf')
    full = list(ctx_toks) + list(tgt_toks)
    ts = len(ctx_toks)
    ids = torch.tensor([full], device=model.device)
    out = model(ids); logits = out.logits[0]
    nll = 0.0; cnt = 0
    for i in range(ts, len(full) - 1):
        lp = torch.log_softmax(logits[i], dim=-1)
        nll += -lp[full[i + 1]].item(); cnt += 1
    del out, logits; torch.cuda.empty_cache()
    if cnt == 0:
        return float('inf'), float('inf')
    mn = nll / cnt
    return math.exp(mn), mn

# ---- multilingual sentence splitter ----
# Splits on Latin (.!?) followed by whitespace AND on CJK end punctuation (。！？) directly.
# Filters out fragments shorter than 4 characters (avoids noise from headers / list markers).
SENT_SPLIT_RE = re.compile(r'(?<=[.!?])\s+|(?<=[\u3002\uff01\uff1f])')
def split_sentences(text):
    parts = SENT_SPLIT_RE.split(text.strip())
    return [p.strip() for p in parts if len(p.strip()) >= 4]

# ---- sentence-aware tokenization ----
# Tokenizes each sentence individually; concatenated token stream may differ slightly from
# whole-text tokenization at sentence boundaries (different BPE merges), but is consistent
# across the three conditions, so within-experiment comparisons are valid.
def tokenize_sentences(sentences):
    out = []
    for s in sentences:
        ids = tokenizer.encode(s, add_special_tokens=False)
        if ids:
            out.append(ids)
    return out

# ---- core: compute ord / sent_shuf / tok_shuf curves at log-spaced ctx ----
def compute_three_condition_curves(text, target_frac=0.5):
    sents = split_sentences(text)
    if len(sents) < 4:
        return None
    sent_ids = tokenize_sentences(sents)
    if len(sent_ids) < 4:
        return None

    sent_lens = [len(s) for s in sent_ids]
    cum = [0]
    for L in sent_lens:
        cum.append(cum[-1] + L)
    n_total = cum[-1]
    if n_total < MIN_DOC_TOK:
        return None

    # Find target sentence: first sentence boundary at-or-after target_frac into the doc.
    target_pos_tok = int(target_frac * n_total)
    target_sent_i = next((i for i in range(1, len(cum)) if cum[i - 1] >= target_pos_tok), None)
    if target_sent_i is None:
        return None

    # Need at least MAX_CTX tokens of prior context before the target sentence.
    if cum[target_sent_i - 1] < MAX_CTX:
        # Try shifting the target sentence forward until enough prior context exists.
        while target_sent_i < len(sent_ids) and cum[target_sent_i - 1] < MAX_CTX:
            target_sent_i += 1
        if target_sent_i >= len(sent_ids):
            return None

    # Build target: TARGET_LEN tokens starting at the target sentence.
    tail = []
    for s in sent_ids[target_sent_i:]:
        tail.extend(s)
        if len(tail) >= TARGET_LEN:
            break
    if len(tail) < TARGET_LEN:
        return None
    target_toks = tail[:TARGET_LEN]

    # Prior sentences and their flat token streams.
    prior_sents = sent_ids[:target_sent_i]
    prior_ord = [t for s in prior_sents for t in s]

    if len(prior_ord) < MAX_CTX:
        return None

    # Sentence-shuffle: permute the order of prior sentences, concatenate.
    rng_sent = random.Random(SEED + 1)
    perm = list(range(len(prior_sents)))
    rng_sent.shuffle(perm)
    prior_sent_shuf = [t for i in perm for t in prior_sents[i]]

    o_ppl, ss_ppl, ts_ppl = [], [], []
    for c in CTX_LENGTHS:
        if c == 0:
            p_ord, _ = ppl_nll([], target_toks)
            o_ppl.append(p_ord)
            ss_ppl.append(p_ord)
            ts_ppl.append(p_ord)
            continue

        # Take last c tokens of each prior stream.
        pfx_ord = prior_ord[-c:]
        pfx_ss = prior_sent_shuf[-c:]

        # Token-shuffle: shuffle the ordered prefix's tokens.
        rng_tok = random.Random(SEED + c)
        pfx_ts = list(pfx_ord); rng_tok.shuffle(pfx_ts)

        p_ord, _ = ppl_nll(pfx_ord, target_toks)
        p_ss, _ = ppl_nll(pfx_ss, target_toks)
        p_ts, _ = ppl_nll(pfx_ts, target_toks)
        o_ppl.append(p_ord); ss_ppl.append(p_ss); ts_ppl.append(p_ts)

    return {
        'context_lengths': list(CTX_LENGTHS),
        'ordered_ppl': o_ppl,
        'sentence_shuffled_ppl': ss_ppl,
        'token_shuffled_ppl': ts_ppl,
        'n_prior_sents': len(prior_sents),
        'n_prior_tokens': len(prior_ord),
        'target_sent_idx': target_sent_i,
    }

print('Pipeline ready')

In [ ]:
# ---- document resolver (mirrors Corpus_Expansion_LongRange_Llama; yields (doc_id, text)) ----
tok_manifest = {}
if TOK_MANIFEST_PATH.exists():
    with open(TOK_MANIFEST_PATH) as f:
        for line in f:
            d = json.loads(line)
            tok_manifest[d['document_id']] = d['file_path']

all_targets = []
if TARGETS_PATH.exists():
    with open(TARGETS_PATH) as f:
        for line in f:
            all_targets.append(json.loads(line))
ce_corpora = {}
for t in all_targets:
    ce_corpora.setdefault(t['corpus_id'], set()).add(t['document_id'])

ML_DATA = DRIVE / 'Data/multilingual_literary'
def literary_docs(lang):
    m = json.loads((ML_DATA / 'manifests' / f'{lang}.json').read_text(encoding='utf-8'))
    for author in m['authors']:
        for t in author['texts']:
            fp = ML_DATA / t['text_path']
            try:
                yield t['text_id'], fp.read_text(encoding='utf-8', errors='replace').strip()
            except Exception:
                continue

BUCKEYE_JSONL = DRIVE / 'Data/buckeye_processed/speaker_concatenated.jsonl'
def buckeye_docs():
    if not BUCKEYE_JSONL.exists():
        return
    with open(BUCKEYE_JSONL) as f:
        for line in f:
            d = json.loads(line)
            txt = d.get('text', '').strip()
            if txt:
                yield d['doc_id'], txt

def fix_ce_path(p):
    return str(p).replace('data/corpus_expansion/clean/',
        '/content/drive/MyDrive/LRTIA/Data/corpus_expansion/')

def ce_docs(corpus_id):
    for doc_id in ce_corpora.get(corpus_id, set()):
        fp = tok_manifest.get(doc_id)
        if fp is None: continue
        abs_fp = Path(fix_ce_path(fp))
        if not abs_fp.exists():
            abs_fp = Path('/content/drive/MyDrive/LRTIA/Data/corpus_expansion') / Path(fp).name
        try:
            yield doc_id, abs_fp.read_text(encoding='utf-8', errors='replace').strip()
        except Exception:
            continue

def get_doc_texts(corpus_id):
    if corpus_id == 'buckeye':
        yield from buckeye_docs()
    elif corpus_id.startswith('literary_'):
        lang = corpus_id.split('_', 1)[1]
        yield from literary_docs(lang)
    else:
        yield from ce_docs(corpus_id)

for c in RUN_CORPORA:
    n = sum(1 for _ in get_doc_texts(c))
    print(f'  {c}: {n} docs found')

In [ ]:
# ---- main run loop ----
for corpus_id in RUN_CORPORA:
    cache_path = BASE / f'{corpus_id}.json'
    if cache_path.exists():
        with open(cache_path) as f: n = len(json.load(f))
        print(f'\n{corpus_id}: cached ({n})'); continue

    docs = list(get_doc_texts(corpus_id))
    if not docs:
        print(f'\n{corpus_id}: no docs'); continue

    print(f'\n{"="*60}\n{corpus_id} ({len(docs)} candidate docs)\n{"="*60}')

    t0 = time.time()
    results = []
    skipped_short = skipped_struct = 0

    for doc_id, text in tqdm(docs, desc=corpus_id):
        for frac in TARGET_FRACS:
            r = compute_three_condition_curves(text, target_frac=frac)
            if r is None:
                # Could be too short OR too few sentences — recheck quickly to attribute.
                full = tokenizer.encode(text, add_special_tokens=False)
                if len(full) < MIN_DOC_TOK:
                    skipped_short += 1
                else:
                    skipped_struct += 1
                continue
            r['corpus_id'] = corpus_id
            r['document_id'] = doc_id
            r['target_id'] = f'{doc_id}__pos{int(frac*100):02d}'
            r['target_frac'] = frac
            results.append(r)

    elapsed = time.time() - t0
    with open(cache_path, 'w') as f:
        json.dump(results, f)
    print(f'  {len(results)} results in {elapsed/60:.1f} min '
          f'({skipped_short} too short, {skipped_struct} too few sentences)')

    if results:
        ord_long = float(np.mean([r['ordered_ppl'][-1] for r in results]))
        ss_long = float(np.mean([r['sentence_shuffled_ppl'][-1] for r in results]))
        ts_long = float(np.mean([r['token_shuffled_ppl'][-1] for r in results]))
        ord_zero = float(np.mean([r['ordered_ppl'][0] for r in results]))
        print(f'  ord[ctx=0]={ord_zero:.2f}  '
              f'ord[ctx={MAX_CTX}]={ord_long:.2f}  '
              f'sent_shuf[ctx={MAX_CTX}]={ss_long:.2f}  '
              f'tok_shuf[ctx={MAX_CTX}]={ts_long:.2f}')
        print(f'  long-range gap (ord vs ts) = {ts_long-ord_long:+.2f}  '
              f'(sent_shuf vs ts) = {ts_long-ss_long:+.2f}  '
              f'(ord vs sent_shuf) = {ss_long-ord_long:+.2f}')

print('\nDone.')